# 03b · Confirm the key models under the corrected evaluator (v2)
**African Credit Scoring Challenge**

`03_features.ipynb` compared 15 feature sets with the original evaluator (**v1**). The corrected evaluator (**v2-nested**, `src/evaluate.py`) keeps each validation fold out of *both* model fitting and threshold selection, so its numbers are the ones reported.

This notebook re-runs only the four models that drive decisions: the two baselines (R0, P0) and the two selected sets (R1, P1), then saves the selected sets for notebook 04. Each full v2 run trains about 100 models.

In [1]:
import sys, json, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, 'src')
from features import build_features                 # corrected features (v2)
from evaluate import Evaluator, log_experiment      # corrected nested evaluator (v2)

warnings.filterwarnings('ignore')
DATA, OUT = Path('.'), Path('outputs')
train = pd.read_csv(DATA / 'Train.csv', parse_dates=['disbursement_date', 'due_date'])
test  = pd.read_csv(DATA / 'Test.csv',  parse_dates=['disbursement_date', 'due_date'])
econ  = pd.read_csv(DATA / 'economic_indicators.csv')
folds = pd.read_csv(OUT / 'folds.csv')

feats = build_features(train, test, econ)
ev = Evaluator(train, test, feats, folds)
print(train.shape, test.shape, feats.shape)

(68654, 16) (18594, 15) (87248, 22)


## 1. The four feature sets
The `[v2]` suffix matters: `log_experiment` replaces rows that share a name, so without it these runs would overwrite the v1 rows.

In [2]:
BASE = ['log_amount_lcu', 'duration', 'is_new', 'lender_share', 'lender_id', 'loan_type']
PRICING = ['daily_rate', 'markup_high', 'markup_excess']
LABEL_FREE_HISTORY = ['hist_n_prior', 'hist_days_since_last', 'hist_log_amount_vs_prior']

R0 = ['log_amount_lcu', 'markup'] + BASE[1:]
SETS = {
    'M3 R0 competition: baseline (notebook 02) [v2]':        R0,
    'M3 R1 competition: R0 + pricing [v2]':             R0 + PRICING,
    'M3 P0 production: no repayment fields [v2]':       BASE,
    'M3 P1 production: P0 + label-free history [v2]':   BASE + LABEL_FREE_HISTORY,
}
for name, cols in SETS.items():
    print(f'{name}: {len(cols)} features')

M3 R0 competition: baseline (notebook 02) [v2]: 7 features
M3 R1 competition: R0 + pricing [v2]: 10 features
M3 P0 production: no repayment fields [v2]: 6 features
M3 P1 production: P0 + label-free history [v2]: 9 features


## 2. Run the full v2 protocol (schemes B, C, D, E)

In [3]:
results = {}
for name, cols in SETS.items():
    start = time.time()
    results[name] = ev.run(name, cols)                 # protocol='v2-nested' by default
    results[name]['seconds'] = round(time.time() - start)
    log_experiment(results[name])
    print(f'   ({results[name]["seconds"] / 60:.1f} minutes)')

M3 R0 competition: baseline (notebook 02) [v2]: F1 B 0.824 ± 0.018 | C 0.808 | D 0.565 | E AUC 0.929 | Ghana flag 11.5%
   (1.2 minutes)


M3 R1 competition: R0 + pricing [v2]: F1 B 0.833 ± 0.010 | C 0.821 | D 0.578 | E AUC 0.949 | Ghana flag 10.5%
   (1.4 minutes)


M3 P0 production: no repayment fields [v2]: F1 B 0.233 ± 0.026 | C 0.224 | D 0.153 | E AUC 0.654 | Ghana flag 15.0%
   (1.0 minutes)


M3 P1 production: P0 + label-free history [v2]: F1 B 0.238 ± 0.036 | C 0.235 | D 0.145 | E AUC 0.663 | Ghana flag 15.1%
   (1.1 minutes)


## 3. Compare v1 and v2 for the same models

In [4]:
log = pd.read_csv(OUT / 'experiments.csv')
pairs = {
    'R0': ('M3 R0 competition: baseline (notebook 02)', 'M3 R0 competition: baseline (notebook 02) [v2]'),
    'R1': ('M3 R1 competition: R0 + pricing',      'M3 R1 competition: R0 + pricing [v2]'),
    'P0': ('M3 P0 production: no repayment fields', 'M3 P0 production: no repayment fields [v2]'),
    'P1': ('M3 P1 production: P0 + label-free history', 'M3 P1 production: P0 + label-free history [v2]'),
}
cols = ['f1_B', 'f1_B_sd', 'f1_C', 'f1_D', 'f1_E_weighted', 'threshold_B']
rows = []
for short, (v1_name, v2_name) in pairs.items():
    for protocol, name in [('v1', v1_name), ('v2-nested', v2_name)]:
        hit = log[log['experiment'] == name]
        if len(hit):
            rows.append({'model': short, 'protocol': protocol, **hit.iloc[-1][cols].to_dict()})
comparison = pd.DataFrame(rows).set_index(['model', 'protocol'])
display(comparison.astype(float).round(3))

f1_B  f1_B_sd   f1_C   f1_D  f1_E_weighted  threshold_B
model protocol                                                           
R0    v1         0.823    0.020  0.810  0.565          0.772         0.50
      v2-nested  0.824    0.018  0.808  0.565          0.709         0.50
R1    v1         0.836    0.012  0.820  0.578          0.768         0.30
      v2-nested  0.833    0.010  0.821  0.578          0.758         0.30
P0    v1         0.222    0.030  0.232  0.153          0.222         0.12
      v2-nested  0.233    0.026  0.224  0.153          0.127         0.12
P1    v1         0.243    0.035  0.227  0.145          0.217         0.11
      v2-nested  0.238    0.036  0.235  0.145          0.297         0.11

In [5]:
for short in pairs:
    if (short, 'v1') in comparison.index and (short, 'v2-nested') in comparison.index:
        v1b, v2b = comparison.loc[(short, 'v1'), 'f1_B'], comparison.loc[(short, 'v2-nested'), 'f1_B']
        sd = comparison.loc[(short, 'v2-nested'), 'f1_B_sd']
        print(f"{short}: scheme B F1 {v1b:.3f} (v1) -> {v2b:.3f} (v2), change {v2b - v1b:+.3f}; "
              f"fold sd {sd:.3f}. {'Within' if abs(v2b - v1b) <= sd else 'Beyond'} the noise.")

R0: scheme B F1 0.823 (v1) -> 0.824 (v2), change +0.000; fold sd 0.018. Within the noise.
R1: scheme B F1 0.836 (v1) -> 0.833 (v2), change -0.003; fold sd 0.010. Within the noise.
P0: scheme B F1 0.222 (v1) -> 0.233 (v2), change +0.011; fold sd 0.026. Within the noise.
P1: scheme B F1 0.243 (v1) -> 0.238 (v2), change -0.005; fold sd 0.036. Within the noise.


## 4. Do the notebook 03 decisions still hold under v2?

In [6]:
r = {k.split(' ')[1]: v for k, v in results.items()}   # 'R0', 'R1', 'P0', 'P1'
def fold_wins(a, b):
    fa = np.array(a['f1_B_folds'].split(), float); fb = np.array(b['f1_B_folds'].split(), float)
    return int((fb > fa).sum())
for parent, child in [('R0', 'R1'), ('P0', 'P1')]:
    a, b = r[parent], r[child]
    print(f"{child} vs {parent}: B {a['f1_B']:.3f} -> {b['f1_B']:.3f} ({b['f1_B'] - a['f1_B']:+.3f}), "
          f"fold wins {fold_wins(a, b)}/5; C {b['f1_C'] - a['f1_C']:+.3f}, D {b['f1_D'] - a['f1_D']:+.3f}, "
          f"E {b['f1_E_weighted'] - a['f1_E_weighted']:+.3f}.")

R1 vs R0: B 0.824 -> 0.833 (+0.010), fold wins 4/5; C +0.013, D +0.013, E +0.049.
P1 vs P0: B 0.233 -> 0.238 (+0.006), fold wins 3/5; C +0.011, D -0.008, E +0.170.


In [7]:
# Assess R1 and P1 under the v2-nested protocol using the results above.
r = {k.split(' ')[1]: v for k, v in results.items()}

def protocol_check(parent, child, tolerance=0.010):
    parent_result, child_result = r[parent], r[child]
    parent_folds = np.array(parent_result['f1_B_folds'].split(), dtype=float)
    child_folds = np.array(child_result['f1_B_folds'].split(), dtype=float)
    fold_wins = int((child_folds > parent_folds).sum())
    b_gain = float(child_result['f1_B'] - parent_result['f1_B'])
    parent_sd = float(parent_result['f1_B_sd'])
    meaningful_b = (b_gain > parent_sd) or (fold_wins >= 4)
    deltas = {
        'C': float(child_result['f1_C'] - parent_result['f1_C']),
        'D': float(child_result['f1_D'] - parent_result['f1_D']),
        'E': float(child_result['f1_E_weighted'] - parent_result['f1_E_weighted']),
    }
    guardrails_ok = all(delta >= -tolerance for delta in deltas.values())
    return {
        'pass': meaningful_b and guardrails_ok,
        'b_gain': b_gain,
        'parent_sd': parent_sd,
        'fold_wins': fold_wins,
        'deltas': deltas,
        'meaningful_b': meaningful_b,
        'guardrails_ok': guardrails_ok,
    }

r1_check = protocol_check('R0', 'R1')
p1_check = protocol_check('P0', 'P1')

print(
    f"R1 vs R0: Scheme B F1 {r['R0']['f1_B']:.3f} -> {r['R1']['f1_B']:.3f} "
    f"({r1_check['b_gain']:+.3f}), {r1_check['fold_wins']}/5 fold wins, "
    f"C {r1_check['deltas']['C']:+.3f}, D {r1_check['deltas']['D']:+.3f}, "
    f"E {r1_check['deltas']['E']:+.3f}; "
    f"{'PASS' if r1_check['pass'] else 'REVIEW'} under v2."
)
print(
    f"P1 vs P0: Scheme B F1 {r['P0']['f1_B']:.3f} -> {r['P1']['f1_B']:.3f} "
    f"({p1_check['b_gain']:+.3f}), {p1_check['fold_wins']}/5 fold wins, "
    f"C {p1_check['deltas']['C']:+.3f}, D {p1_check['deltas']['D']:+.3f}, "
    f"E {p1_check['deltas']['E']:+.3f}; "
    f"{'PASS' if p1_check['pass'] else 'REVIEW'} under v2."
)
r1_conclusion = (
    "R1 remains supported as the competition selection."
    if r1_check['pass']
    else "R1 requires further review before it can be treated as the competition selection."
)
p1_conclusion = (
    "P1 passes the v2 production comparison."
    if p1_check['pass']
    else (
        f"P1 does not pass the v2 Scheme B improvement rule "
        f"(gain must exceed parent fold SD {p1_check['parent_sd']:.3f} "
        f"or win at least 4/5 folds), so it remains a provisional production challenger. "
    )
)
p1_auc = float(r['P1']['roc_auc_B'])
p0_auc = float(r['P0']['roc_auc_B'])
ranking_conclusion = (
    f"P1 ranks borrowers more clearly than P0 (ROC AUC {p1_auc:.3f} vs {p0_auc:.3f}), "
    "which supports its use for the production track's risk bands."
    if p1_auc > p0_auc
    else (
        f"P1 does not improve ROC AUC over P0 ({p1_auc:.3f} vs {p0_auc:.3f}); "
        "its ranking advantage should therefore be re-examined before relying on risk bands."
    )
)
if r1_check['pass'] and p1_check['pass']:
    overall_conclusion = "Both R1 and P1 are confirmed by their respective v2 checks."
elif r1_check['pass'] and not p1_check['pass']:
    overall_conclusion = (
        "R1 is confirmed; P1 remains a provisional production challenger rather than a passed selection."
    )
elif not r1_check['pass'] and p1_check['pass']:
    overall_conclusion = "P1 is confirmed; R1 requires further review."
else:
    overall_conclusion = "Both selections require further review under the v2 protocol."

print(
    f"Therefore, {r1_conclusion} {p1_conclusion} "
    f"{ranking_conclusion} {overall_conclusion}"
)


R1 vs R0: Scheme B F1 0.824 -> 0.833 (+0.010), 4/5 fold wins, C +0.013, D +0.013, E +0.049; PASS under v2.
P1 vs P0: Scheme B F1 0.233 -> 0.238 (+0.006), 3/5 fold wins, C +0.011, D -0.008, E +0.170; REVIEW under v2.
Therefore, R1 remains supported as the competition selection. P1 does not pass the v2 Scheme B improvement rule (gain must exceed parent fold SD 0.026 or win at least 4/5 folds), so it remains a provisional production challenger.  P1 ranks borrowers more clearly than P0 (ROC AUC 0.857 vs 0.798), which supports its use for the production track's risk bands. R1 is confirmed; P1 remains a provisional production challenger rather than a passed selection.


## 5. Save the selected feature sets for notebook 04

In [8]:
selected = {
    'competition': SETS['M3 R1 competition: R0 + pricing [v2]'],
    'production':  SETS['M3 P1 production: P0 + label-free history [v2]'],
}
(OUT / 'feature_sets.json').write_text(json.dumps(selected, indent=2))
assert not {'hist_n_known', 'hist_n_defaults', 'hist_default_rate'} & set(selected['production'])
assert not {'markup', 'daily_rate', 'markup_high', 'markup_excess'} & set(selected['production'])
print(json.dumps(selected, indent=2))

{
  "competition": [
    "log_amount_lcu",
    "markup",
    "duration",
    "is_new",
    "lender_share",
    "lender_id",
    "loan_type",
    "daily_rate",
    "markup_high",
    "markup_excess"
  ],
  "production": [
    "log_amount_lcu",
    "duration",
    "is_new",
    "lender_share",
    "lender_id",
    "loan_type",
    "hist_n_prior",
    "hist_days_since_last",
    "hist_log_amount_vs_prior"
  ]
}
